# 📊 Modul 1: Chunking & Embedding — Finanzberichte
> **RAG-Framework** · Phase 2 · Echte Unternehmensdaten

Dieses Notebook verarbeitet **~900 Finanz-PDFs** (Konzernabschlüsse, Geschäftsberichte,
Kapitalmarktmitteilungen) und baut daraus eine durchsuchbare Wissensbasis auf.

---
### Inhalt
1. Setup & Imports
2. PDF-Inventar & Klassifikation
3. Batch-Verarbeitung & Text-Extraktion
4. Chunking-Strategie für Finanzdokumente
5. Embedding & Vektordatenbank
6. Retrieval-Test mit echten Fragen
7. Zusammenfassung & Nächste Schritte


---
## 1. Setup & Imports

In [ ]:
# ── Installation (nur beim ersten Ausführen) ───────────────────────────────
# Raute entfernen und einmal ausführen:
# !pip install langchain langchain-community langchain-text-splitters \
#              sentence-transformers pypdf chromadb scikit-learn \
#              matplotlib seaborn pandas numpy python-dotenv tqdm --quiet

In [ ]:
import os
import re
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from IPython.display import display, Markdown

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

warnings.filterwarnings('ignore')
load_dotenv()

# Plotting-Style
plt.style.use('dark_background')
ACCENT  = '#00f5d4'
ACCENT2 = '#7b2fff'
ACCENT3 = '#ff6b35'

# ── Pfade ──────────────────────────────────────────────────────────────────
BASE_DIR      = Path('..')           # Projektroot
DATA_RAW      = BASE_DIR / 'data' / 'raw'
DATA_PROC     = BASE_DIR / 'data' / 'processed'
CHROMA_DIR    = BASE_DIR / 'data' / 'chroma_db'
PROGRESS_FILE = DATA_PROC / 'processing_progress.json'

# Verzeichnisse anlegen falls nicht vorhanden
DATA_PROC.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup abgeschlossen')
print(f'   Rohdaten  : {DATA_RAW.resolve()}')
print(f'   Output    : {DATA_PROC.resolve()}')

---
## 2. PDF-Inventar & Klassifikation

Alle PDFs werden eingelesen, nach Typ klassifiziert und Jahre aus den Dateinamen extrahiert.

In [ ]:
# ── Dokumenttyp & Metadaten aus Dateiname erkennen ────────────────────────
def classify_document(filename: str) -> dict:
    name = filename.lower()

    if 'konzernabschluss' in name or 'rechnungslegung' in name:
        doc_type = 'Konzernabschluss'
    elif 'geschaeftsbericht' in name or 'geschäftsbericht' in name:
        doc_type = 'Geschäftsbericht'
    elif 'kapitalmarkt' in name or 'mitteilung' in name:
        doc_type = 'Kapitalmarktmitteilung'
    elif any(x in name for x in ['quartal', 'q1', 'q2', 'q3', 'q4']):
        doc_type = 'Quartalsbericht'
    elif 'risiko' in name:
        doc_type = 'Risikoanalyse'
    elif any(x in name for x in ['pruefer', 'prüfer', 'audit']):
        doc_type = 'Prüfungsbericht'
    else:
        doc_type = 'Sonstiges'

    years = re.findall(r'\b(20[0-2][0-9]|199[0-9])\b', filename)
    year_start = int(years[0])  if years          else None
    year_end   = int(years[-1]) if len(years) > 1 else year_start

    return {
        'doc_type'   : doc_type,
        'year_start' : year_start,
        'year_end'   : year_end,
        'fiscal_year': f'{year_start}/{year_end}' if year_start != year_end else str(year_start),
    }

# Inventar aufbauen
pdf_files = sorted(DATA_RAW.glob('*.pdf'))
inventory = []
for pdf in pdf_files:
    meta = classify_document(pdf.name)
    inventory.append({
        'filename' : pdf.name,
        'path'     : str(pdf),
        'size_kb'  : round(pdf.stat().st_size / 1024, 1),
        **meta
    })

df_inv = pd.DataFrame(inventory)
df_inv.to_csv(DATA_PROC / 'inventar.csv', index=False)

print(f'📂 PDFs gefunden    : {len(df_inv)}')
print(f'   Gesamtgröße     : {df_inv["size_kb"].sum()/1024:.1f} MB')
print(f'   Zeitraum        : {df_inv["year_start"].min()} – {df_inv["year_end"].max()}')
print(f'\nDokumenttypen:')
print(df_inv['doc_type'].value_counts().to_string())

In [ ]:
# ── Inventar visualisieren ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('PDF-Inventar Übersicht', color='white', fontsize=14)
colors = [ACCENT, ACCENT2, ACCENT3, '#ffd166', '#06d6a0', '#ef476f', '#118ab2']

# Dokumenttypen
tc = df_inv['doc_type'].value_counts()
axes[0].barh(tc.index, tc.values, color=colors[:len(tc)], alpha=0.85)
axes[0].set_title('Dokumenttypen', color='white')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#333')

# Dokumente pro Jahr
yc = df_inv['year_end'].value_counts().sort_index()
axes[1].bar(yc.index.astype(str), yc.values, color=ACCENT, alpha=0.85)
axes[1].set_title('Dokumente pro Jahr', color='white')
axes[1].tick_params(rotation=45)

# Dateigrößen
axes[2].hist(df_inv['size_kb'], bins=20, color=ACCENT2, alpha=0.85)
axes[2].axvline(df_inv['size_kb'].mean(), color=ACCENT, linestyle='--',
                label=f'Ø {df_inv["size_kb"].mean():.0f} KB')
axes[2].set_title('Dateigrößen (KB)', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig(DATA_PROC / 'inventar_uebersicht.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('💾 Gespeichert: data/processed/inventar_uebersicht.png')

---
## 3. Batch-Verarbeitung & Text-Extraktion

**Bereits verarbeitete PDFs werden übersprungen** — du kannst jederzeit unterbrechen
und das Notebook neu starten, ohne von vorne anzufangen.

In [ ]:
# ── Fortschritt laden ────────────────────────────────────────────────────
def load_progress() -> dict:
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'processed': [], 'failed': [], 'last_run': None}

def save_progress(prog: dict):
    prog['last_run'] = datetime.now().isoformat()
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(prog, f, ensure_ascii=False, indent=2)

progress     = load_progress()
already_done = set(progress['processed'])

print(f'📋 Fortschritt geladen:')
print(f'   Bereits verarbeitet : {len(already_done)} Dateien')
print(f'   Fehlgeschlagen      : {len(progress["failed"])} Dateien')
print(f'   Verbleibend         : {len(pdf_files) - len(already_done)} Dateien')

In [ ]:
# ── Konfiguration ─────────────────────────────────────────────────────────
# Für ersten Test: 10 PDFs | Für alle: None
BATCH_LIMIT   = 10    # ← auf None setzen für alle ~900 PDFs
MIN_TEXT_LEN  = 100   # Seiten kürzer als N Zeichen überspringen
SAVE_INTERVAL = 20    # Fortschritt alle N Dateien sichern

print(f'⚙️  Konfiguration:')
print(f'   Batch-Limit  : {BATCH_LIMIT if BATCH_LIMIT else "Alle PDFs"}')
print(f'   Min-Textlänge: {MIN_TEXT_LEN} Zeichen/Seite')

In [ ]:
# ── Batch-PDF-Extraktion ──────────────────────────────────────────────────
def extract_pdf(pdf_path: Path, meta: dict) -> list:
    try:
        pages = PyPDFLoader(str(pdf_path)).load()
        docs  = []
        for page in pages:
            text = page.page_content.strip()
            if len(text) < MIN_TEXT_LEN:
                continue
            page.metadata.update({
                'source_file' : pdf_path.name,
                'doc_type'    : meta['doc_type'],
                'year_start'  : meta['year_start'],
                'year_end'    : meta['year_end'],
                'fiscal_year' : meta['fiscal_year'],
            })
            docs.append(page)
        return docs
    except Exception as e:
        return []

# Zu verarbeitende Dateien
to_process = [r for _, r in df_inv.iterrows() if r['filename'] not in already_done]
if BATCH_LIMIT:
    to_process = to_process[:BATCH_LIMIT]

print(f'🔄 Verarbeite {len(to_process)} PDFs...\n')

all_documents = []
pages_total   = 0
failed_count  = 0

for i, row in enumerate(tqdm(to_process, desc='PDF-Extraktion')):
    docs = extract_pdf(Path(row['path']), row)
    if docs:
        all_documents.extend(docs)
        pages_total += len(docs)
        progress['processed'].append(row['filename'])
    else:
        failed_count += 1
        progress['failed'].append(row['filename'])
    if (i + 1) % SAVE_INTERVAL == 0:
        save_progress(progress)

save_progress(progress)

print(f'\n✅ Extraktion abgeschlossen:')
print(f'   Seiten extrahiert  : {pages_total:,}')
print(f'   Fehlgeschlagen     : {failed_count}')
print(f'   Gesamt verarbeitet : {len(progress["processed"])} / {len(pdf_files)} PDFs')

---
## 4. Chunking-Strategie für Finanzdokumente

Finanztexte haben besondere Anforderungen:
- Kennzahlen brauchen ihren Kontext (z.B. "EBITDA 2023: 410 Mio. €")
- Tabellenzellen sollen nicht mitten im Satz getrennt werden
- Jahresangaben müssen im Chunk sichtbar bleiben

In [ ]:
# ── Finanz-optimierter Chunker ────────────────────────────────────────────
CHUNK_SIZE    = 600
CHUNK_OVERLAP = 80

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', '; ', ', ', ' ', ''],
    length_function=len,
)

print(f'🔪 Chunking:')
print(f'   Chunk-Größe  : {CHUNK_SIZE} Zeichen')
print(f'   Überlappung  : {CHUNK_OVERLAP} Zeichen')
print(f'   Input-Seiten : {len(all_documents):,}')

chunks = splitter.split_documents(all_documents)
chunk_lengths = [len(c.page_content) for c in chunks]

print(f'\n✅ Chunking abgeschlossen:')
print(f'   Chunks erstellt  : {len(chunks):,}')
print(f'   Ø Chunk-Größe    : {np.mean(chunk_lengths):.0f} Zeichen')
print(f'   Min / Max        : {np.min(chunk_lengths)} / {np.max(chunk_lengths)} Zeichen')

In [ ]:
# ── Chunk-Qualität visualisieren ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Chunk-Qualitätsanalyse', color='white', fontsize=13)

for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#333')

# Chunk-Längenverteilung
axes[0].hist(chunk_lengths, bins=30, color=ACCENT, alpha=0.85)
axes[0].axvline(np.mean(chunk_lengths), color='white', linestyle='--',
                label=f'Ø {np.mean(chunk_lengths):.0f}')
axes[0].axvline(CHUNK_SIZE, color=ACCENT3, linestyle=':', label=f'Max {CHUNK_SIZE}')
axes[0].set_title('Chunk-Längen', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)

# Chunks pro Dokumenttyp
type_counts = pd.Series([c.metadata.get('doc_type','?') for c in chunks]).value_counts()
axes[1].barh(type_counts.index, type_counts.values,
             color=[ACCENT, ACCENT2, ACCENT3, '#ffd166', '#06d6a0'][:len(type_counts)],
             alpha=0.85)
axes[1].set_title('Chunks pro Dokumenttyp', color='white')

# Chunks pro Jahr
years = [c.metadata.get('year_end') for c in chunks if c.metadata.get('year_end')]
year_counts = pd.Series(years).value_counts().sort_index()
axes[2].bar(year_counts.index.astype(str), year_counts.values, color=ACCENT2, alpha=0.85)
axes[2].set_title('Chunks pro Jahr', color='white')
axes[2].tick_params(rotation=45)

plt.tight_layout()
plt.savefig(DATA_PROC / 'chunk_analyse.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── 3 Beispiel-Chunks inspizieren ─────────────────────────────────────────
print('🔍 Beispiel-Chunks aus den Finanzdokumenten:')
print('='*65)
for i, chunk in enumerate(random.sample(chunks, min(3, len(chunks)))):
    print(f'\n[Chunk {i+1}]')
    print(f'  Datei    : {chunk.metadata.get("source_file","?")[:58]}')
    print(f'  Typ      : {chunk.metadata.get("doc_type","?")}')
    print(f'  Jahr     : {chunk.metadata.get("fiscal_year","?")}')
    print(f'  Länge    : {len(chunk.page_content)} Zeichen')
    print(f'  Text     : {chunk.page_content[:220]}...')
    print('-'*65)

---
## 5. Embedding & Vektordatenbank

**BGE-M3** — mehrsprachig, SOTA, läuft vollständig lokal. Kein API-Key nötig.

In [ ]:
# ── Embedding-Modell laden ────────────────────────────────────────────────
print('⏳ Lade Embedding-Modell BGE-M3...')
print('   (Erster Start: ~2 GB Download — danach gecacht)')

embedder = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
)

test_vec = embedder.embed_query('Jahresüberschuss 2023')
print(f'\n✅ Modell bereit — Dimensionen: {len(test_vec)}')

In [ ]:
# ── Vektordatenbank aufbauen ──────────────────────────────────────────────
EMBED_BATCH = 100
print(f'🗄️  Indexiere {len(chunks):,} Chunks in Batches von {EMBED_BATCH}...')
start = time.time()

vectorstore = Chroma.from_documents(
    chunks[:EMBED_BATCH], embedder,
    persist_directory=str(CHROMA_DIR),
    collection_name='finanzberichte'
)

for i in tqdm(range(0, len(chunks) - EMBED_BATCH, EMBED_BATCH), desc='Embeddings'):
    vectorstore.add_documents(chunks[EMBED_BATCH + i : EMBED_BATCH + i + EMBED_BATCH])

elapsed = time.time() - start
print(f'\n✅ Vektordatenbank fertig:')
print(f'   Chunks indexiert : {len(chunks):,}')
print(f'   Dauer            : {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'   Gespeichert in   : {CHROMA_DIR}')

---
## 6. Retrieval-Test mit echten Finanzfragen

In [ ]:
# ── Vordefinierte Testfragen ──────────────────────────────────────────────
TEST_QUERIES = [
    'Wie hat sich der Jahresüberschuss über die Jahre entwickelt?',
    'Welche Risiken wurden in den Geschäftsberichten identifiziert?',
    'Wie hoch war die Eigenkapitalrendite?',
    'Was sind die strategischen Ziele des Unternehmens?',
    'Wie hat sich die Dividende entwickelt?',
    'Welche Akquisitionen wurden durchgeführt?',
    'Wie hat sich das EBITDA über die Jahre verändert?',
]

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

print('🔍 Retrieval-Test')
print('='*65)
for query in TEST_QUERIES[:3]:
    print(f'\n❓ {query}')
    results = retriever.invoke(query)
    for j, r in enumerate(results[:2]):
        print(f'  [{j+1}] {r.metadata.get("source_file","?")[:50]}')
        print(f'       Jahr: {r.metadata.get("fiscal_year","?")} | Typ: {r.metadata.get("doc_type","?")}')
        print(f'       {r.page_content[:160]}...')
    print('-'*65)

In [ ]:
# ── Eigene Frage stellen ──────────────────────────────────────────────────
# ← Hier deine eigene Frage eintragen:
MEINE_FRAGE = 'Wie hat sich der Umsatz seit 2010 entwickelt?'

results = retriever.invoke(MEINE_FRAGE)
print(f'❓ Frage: {MEINE_FRAGE}')
print(f'\n📄 Top-{len(results)} Ergebnisse:\n')
for i, r in enumerate(results):
    print(f'── Ergebnis {i+1} ──────────────────────────────────')
    print(f'  Quelle        : {r.metadata.get("source_file","?")[:55]}')
    print(f'  Dokumenttyp   : {r.metadata.get("doc_type","?")}')
    print(f'  Geschäftsjahr : {r.metadata.get("fiscal_year","?")}')
    print(f'  Seite         : {r.metadata.get("page","?")}')
    print(f'  Text          : {r.page_content[:300]}')
    print()

---
## 7. Zusammenfassung & Nächste Schritte

In [ ]:
# ── Abschlussbericht ──────────────────────────────────────────────────────
total_proc = len(progress['processed'])
pct_done   = total_proc / max(len(pdf_files), 1) * 100

display(Markdown(f"""
## 📊 Verarbeitungsstand

| Kennzahl | Wert |
|---|---|
| PDFs gesamt | `{len(pdf_files)}` |
| PDFs verarbeitet | `{total_proc}` ({pct_done:.1f}%) |
| PDFs fehlgeschlagen | `{len(progress['failed'])}` |
| Seiten extrahiert | `{pages_total:,}` |
| Chunks erstellt | `{len(chunks):,}` |
| Chunk-Größe | `{CHUNK_SIZE}` Zeichen (Overlap `{CHUNK_OVERLAP}`) |
| Embedding-Modell | `BAAI/bge-m3` (lokal, multilingual) |
| Vektordatenbank | `ChromaDB` lokal in `data/chroma_db/` |

## 🚀 Nächste Schritte

1. **`BATCH_LIMIT = None`** setzen → alle ~900 PDFs in einem Durchlauf verarbeiten
2. **Notebook 02**: Advanced RAG — Re-Ranking, HyDE, Query-Transformation
3. **Notebook 03**: GraphRAG — Entitäten & Relationen aus Finanztexten extrahieren
4. **Notebook 04**: Agenten — Zeitreihen-Analysen über alle Geschäftsjahre automatisieren
"""))

save_progress(progress)
print('💾 Fortschritt gespeichert — jederzeit weitermachbar!')